# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Model Training
This is the eighth stage of the pipeline. In this notebook, we establish our **Baseline Machine Learning Models**. We will train the algorithms on the TF-IDF feature matrices generated in the previous stage.

## Strict Training Data Rules
To guarantee zero data leakage, this notebook will **strictly load only the training data (`X_train_tfidf.npz`, `y_train.csv`)**. The validation and test datasets are deliberately ignored in this notebook. They are reserved exclusively for the hyperparameter tuning and final model evaluation stages.


# 2. MODEL TRAINING OBJECTIVES

Our goals for this stage are to:
- Train baseline Machine Learning models on high-dimensional, sparse TF-IDF text features.
- Fulfill the requirement of using at least two different algorithms, including at least one Boosting algorithm.
- Establish reproducible model artifacts for later tuning.
- Track basic metadata like training time and feature dimensionality.
- Preserve validation and test data for completely unbiased evaluation later.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
from scipy import sparse
import joblib
import time
import json
import warnings

# Machine Learning Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import xgboost as xgb

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR = BASE_REPORTS_DIR / "model_training"

# Input paths
X_TRAIN_PATH = TFIDF_DIR / "X_train_tfidf.npz"
Y_TRAIN_PATH = TFIDF_DIR / "y_train.csv"

# Global Config
RANDOM_STATE = 42

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Artifact Directory: {TFIDF_DIR}")


Artifact Directory: /home/zapp/Kampus/PM-NEW/data/processed/tfidf


In [3]:
# 5. LOAD TF-IDF ARTIFACTS

if not X_TRAIN_PATH.exists() or not Y_TRAIN_PATH.exists():
    raise FileNotFoundError(f"FAIL: Training artifacts not found in {TFIDF_DIR}. Please run 06_tfidf.ipynb first.")

# We intentionally DO NOT load validation or test data to prevent leakage.
print("Loading Training Data...")
X_train = sparse.load_npz(X_TRAIN_PATH)
y_train_df = pd.read_csv(Y_TRAIN_PATH)
y_train = y_train_df.iloc[:, 0]  # Get the first column as a Series

print(f"X_train shape: {X_train.shape} | Sparse Matrix")
print(f"y_train shape: {y_train.shape} | Target Labels")


Loading Training Data...
X_train shape: (30195, 1959527) | Sparse Matrix
y_train shape: (30195,) | Target Labels


In [4]:
# 6. VALIDATE FEATURE AND LABEL ALIGNMENT

if X_train.shape[0] != y_train.shape[0]:
    raise ValueError(f"CRITICAL ERROR: Misalignment! X_train has {X_train.shape[0]} rows, but y_train has {y_train.shape[0]} rows.")

print("Feature and Label alignment validated successfully.")


Feature and Label alignment validated successfully.


In [5]:
# 7. INSPECT TARGET LABELS

# Display class distribution for the training set
class_counts = y_train.value_counts()
class_percentages = y_train.value_counts(normalize=True) * 100

print("Target Labels in Training Data:")
for class_name, count in class_counts.items():
    print(f"- {class_name:<15}: {count} ({class_percentages[class_name]:.2f}%)")
    
# XGBoost requires target classes to be integers starting from 0.
# We will create a label mapping and encode y_train dynamically during the XGBoost step.
label_classes = sorted(y_train.unique())
label_mapping = {label: idx for idx, label in enumerate(label_classes)}
print(f"\nInternal Label Mapping for XGBoost: {label_mapping}")


Target Labels in Training Data:
- normal         : 20192 (66.87%)
- hate_speech    : 5125 (16.97%)
- harassment     : 3659 (12.12%)
- insult         : 1219 (4.04%)

Internal Label Mapping for XGBoost: {'harassment': 0, 'hate_speech': 1, 'insult': 2, 'normal': 3}


In [6]:
# 8. DEFINE MODELS

# We define 3 Baseline Models suitable for Sparse TF-IDF matrices:

# 1. Logistic Regression: Strong, interpretable baseline for text data.
model_lr = LogisticRegression(
    max_iter=1000, 
    random_state=RANDOM_STATE,
    class_weight='balanced' # Often helps in highly imbalanced cyberbullying datasets
)

# 2. Linear Support Vector Machine: Highly effective for finding hyperplanes in sparse, high-dimensional spaces.
model_svm = LinearSVC(
    max_iter=2000,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    dual=False # Recommended when n_samples > n_features
)

# 3. XGBoost: The required boosting algorithm. 
# Chosen because it natively supports scipy.sparse matrices without crashing RAM.
model_xgb = xgb.XGBClassifier(
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cpu' # Reverted to CPU to prevent VRAM OOM # Highly optimized for sparse data
)

models_dict = {
    'logistic_regression_baseline': model_lr,
    'linear_svm_baseline': model_svm,
    'xgboost_baseline': model_xgb
}

print("Baseline models defined.")


Baseline models defined.


In [7]:
# 9. DEFINE TRAINING WORKFLOW

def train_baseline_model(model_name, model, X, y):
    print(f"\n--- Training {model_name} ---")
    
    # XGBoost strict label encoding requirement
    if 'xgboost' in model_name:
        y_encoded = y.map(label_mapping)
    else:
        y_encoded = y

    start_time = time.time()
    
    # Fit the model
    model.fit(X, y_encoded)
    
    end_time = time.time()
    duration = end_time - start_time
    
    print(f"Training completed in {duration:.4f} seconds.")
    
    metadata = {
        "model_name": model_name,
        "algorithm": type(model).__name__,
        "training_samples": X.shape[0],
        "feature_dimensions": X.shape[1],
        "training_duration_seconds": duration,
        "random_state": getattr(model, 'random_state', 'N/A')
    }
    
    return model, metadata

print("Training workflow initialized.")


Training workflow initialized.


In [8]:
# 10. TRAIN BASELINE MODELS

trained_models = {}
training_metadata = []

for name, model in models_dict.items():
    trained_model, meta = train_baseline_model(name, model, X_train, y_train)
    trained_models[name] = trained_model
    training_metadata.append(meta)

print("\nAll baseline models have been successfully trained.")



--- Training logistic_regression_baseline ---
Training completed in 71.8748 seconds.

--- Training linear_svm_baseline ---
Training completed in 16.6672 seconds.

--- Training xgboost_baseline ---


XGBoostError: [22:56:16] /__w/xgboost/xgboost/src/common/device_vector.cu:53: Memory allocation error on worker 0: std::bad_alloc: cudaErrorMemoryAllocation: out of memory
- Free memory: 3.86755GB
- Requested memory: 4.20275GB
- Reserved by the pool: 0B
- Used by the pool: 0B

Stack trace:
  [bt] (0) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x2c69bc) [0x7f4ebe2c69bc]
  [bt] (1) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0xb9ab86) [0x7f4ebeb9ab86]
  [bt] (2) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0xc5e805) [0x7f4ebec5e805]
  [bt] (3) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x1296c07) [0x7f4ebf296c07]
  [bt] (4) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.so(+0x12970bc) [0x7f4ebf2970bc]
  [bt] (5) /home/zapp/Kampus/PM-NEW/venv/lib/python3.14/site-packages/xgboost/lib/libxg

In [ ]:
# 11. TRAIN ADDITIONAL TARGET MODELS

# This project document focuses primarily on `cyberbullying_type` as the target column.
# If severity classification is required later, the same workflow (Cells 8-10) 
# can be duplicated targeting a `y_train_severity.csv` file.

print("Additional target models skipped (focusing on primary Cyberbullying Type classification).")


In [ ]:
# 12. PERFORM TRAINING SANITY CHECKS

print("Running sanity checks...")

for name, model in trained_models.items():
    # Check if model has classes_ attribute (which means it's fitted)
    if hasattr(model, 'classes_'):
        print(f"[OK] {name} is fitted and recognizes {len(model.classes_)} classes.")
    else:
        print(f"[ERROR] {name} does not appear to be fitted properly.")


In [ ]:
# 13. SAVE TRAINED MODELS

for name, model in trained_models.items():
    model_path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(model, model_path)
    print(f"Saved: {model_path}")
    
# Save label mapping for XGBoost decoding later
mapping_path = MODELS_DIR / "xgboost_label_mapping.json"
with open(mapping_path, 'w') as f:
    json.dump(label_mapping, f, indent=4)
print(f"Saved: {mapping_path}")


In [ ]:
# 14. SAVE TRAINING METADATA

meta_path = REPORTS_DIR / "model_training_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(training_metadata, f, indent=4)

print(f"Training metadata saved to {meta_path}")


# 15. TRAINING SUMMARY

### Models Trained
1. **Logistic Regression** (`logistic_regression_baseline.pkl`)
2. **Linear SVM** (`linear_svm_baseline.pkl`)
3. **XGBoost** (`xgboost_baseline.pkl`)

### Training Data
- **Target**: `cyberbullying_type`
- **Samples**: ~30,000 (70% of dataset)
- **Features**: Refer to TF-IDF matrix dimensions.

### Data Leakage Validation
- Only `X_train_tfidf.npz` and `y_train.csv` were loaded.
- Validation and Test datasets were completely isolated from this process.

### Next Step
The baseline models have been successfully established. They will now be used in `notebooks/08_hyperparameter_tuning.ipynb` where we will use the Validation dataset to optimize their performance.


# 16. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the TF-IDF artifacts exist in: `data/processed/tfidf/`
3. Ensure the TF-IDF stage has been completed: `notebooks/06_tfidf.ipynb`
4. Open: `notebooks/07_model_training.ipynb`
5. Verify the configured input artifact paths in Step 4.
6. Review the selected models (Logistic Regression, LinearSVC, XGBoost) in Step 8.
7. Run the notebook from the first cell to the last cell.
8. Pay attention to the training duration printed in Step 10. XGBoost might take slightly longer depending on the matrix size.
9. Review the training sanity checks in Step 12 to ensure all models are properly fitted.
10. Confirm that the model `.pkl` files were generated in the `models/` directory.
11. Proceed to: `notebooks/08_hyperparameter_tuning.ipynb`
